In [1]:
import torch
import numpy as np
from src import gen_hypercubes, intersection_matrix, count

divide edges into groups by which coordinate differs

Consider the formula 

In [3]:
def get_ith_intersections(d, h, edge_set):
    for i in range(d):
        mask = edge_set[:,i] != edge_set[:,i+d]
        i_edges = edge_set[mask]
        print(f"{i}: {count(d,h,i_edges)}")

In [14]:


t1_5_6 = torch.tensor([[1,1,1,3,3,-4,0],
                        [-2,-2,-2,3,3,-1,0],
                        [3,3,3,1,1,-4,0],
                        [-1,-1,-1,3,3,6,0],
                        [3,3,3,1,1,8,0]]).double()

t2_5_6 = torch.tensor([[2,2,2,-1,-1,-3,0],
                        [2,2,2,3,3,1,0],
                        [1,1,1,-3,-3,-4,0],
                        [1,1,1,2,2,-4,0],
                        [2,2,2,-1,-1,5,0]]).double()

#h = torch.tensor([1,1,1,3,3,6,0]).double()
#h = torch.tensor([[1,1,1,-3,3,-4,0],
#  [-2,2,2,3,3,-1,0]]).double()


t_6_7 = torch.zeros([6,8]).double()
t_6_7[0,0]=1
t_6_7[1:,1:] = t2_5_6


t_7_8 = torch.zeros([7,9]).double()
t_7_8[0,0]=1
t_7_8[1:,1:] = t_6_7


h = t_7_8
d=8

hp1, hp2 = gen_hypercubes(d)

edge_set = torch.tensor(np.concatenate((hp1, hp2)).T).double()
print(h)
print(count(d,h, edge_set), 2**(d-1)*d)
get_ith_intersections(d,h,edge_set)

tensor([[ 1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  1.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 0.,  0.,  2.,  2.,  2., -1., -1., -3.,  0.],
        [ 0.,  0.,  2.,  2.,  2.,  3.,  3.,  1.,  0.],
        [ 0.,  0.,  1.,  1.,  1., -3., -3., -4.,  0.],
        [ 0.,  0.,  1.,  1.,  1.,  2.,  2., -4.,  0.],
        [ 0.,  0.,  2.,  2.,  2., -1., -1.,  5.,  0.]], dtype=torch.float64)
tensor(1024) 1024
0: 128
1: 128
2: 128
3: 128
4: 128
5: 128
6: 128
7: 128


Attempt to see - use two vectors, one for just getting a lot of last dimension and one for more even spread. Sort dimensions from list to most needed, randomly apply vectors and signs to get the most possible new. 

In [15]:
4<<1

8

In [29]:
def all_tuple_iterate(d):
    for i in range(2**d):
        yield tuple(-1 if (i>>j)%2==0 else 1 for j in range(d))

v=[1,1.99,3]

d=len(v)
l=[]
for t in all_tuple_iterate(d):
    l.append([t, sum([t[i]*v[i] for i in range(d)])])
l_sorted = sorted(l, key=lambda item : item[1])
for item in l_sorted:
    print(item)

[(-1, -1, -1), -5.99]
[(1, -1, -1), -3.99]
[(-1, 1, -1), -2.01]
[(1, 1, -1), -0.009999999999999787]
[(-1, -1, 1), 0.009999999999999787]
[(1, -1, 1), 2.01]
[(-1, 1, 1), 3.99]
[(1, 1, 1), 5.99]


Hm. 

Could we somehow... get a specific sequence of differences? Does applying a negative to two coefficients get the same thing as doing so one at a time?